In [1]:
!pip install kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [3]:
import kagglehub

path = kagglehub.competition_download('games-rating')

print("Path: ", path)

100%|██████████| 797k/797k [00:00<00:00, 46.3MB/s]

Extracting files...
Path:  /root/.cache/kagglehub/competitions/games-rating


In [4]:
# поменять путь
!mv /root/.cache/kagglehub/competitions/games-rating games-rating

In [41]:
!pip install catboost -q

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')

In [52]:
train_df = pd.read_csv('games-rating/train_data.csv')
test_df = pd.read_csv('games-rating/test_data.csv')
sample_sub = pd.read_csv('games-rating/sample_submition.csv')

def clean_number(value):
    if pd.isna(value):
        return np.nan
    try:
        cleaned = str(value).replace(',', '.').replace(' ', '').replace('"', '')
        return float(cleaned)
    except:
        return np.nan

train_df['Rating Average'] = train_df['Rating Average'].apply(clean_number)
train_df = train_df.dropna(subset=['Rating Average'])

numeric_cols = ['Year Published', 'Min Players', 'Max Players', 'Play Time',
                'Min Age', 'Users Rated', 'BGG Rank', 'Complexity Average', 'Owned Users']

for col in numeric_cols:
    train_df[col] = train_df[col].apply(clean_number)
    test_df[col] = test_df[col].apply(clean_number)
    median_val = train_df[col].median()
    train_df[col] = train_df[col].fillna(median_val)
    test_df[col] = test_df[col].fillna(median_val)

In [53]:
def create_advanced_features(df):
    df = df.copy()

    df['Player Range'] = df['Max Players'] - df['Min Players']
    df['Avg Players'] = (df['Min Players'] + df['Max Players']) / 2
    df['Years Since'] = (2021 - df['Year Published']).clip(lower=1)

    df['Popularity'] = df['Users Rated'] / df['Years Since']
    df['Owners per Rating'] = df['Owned Users'] / (df['Users Rated'] + 1)
    df['Rating per Owner'] = df['Users Rated'] / (df['Owned Users'] + 1)
    df['BGG Score'] = 1 / (df['BGG Rank'] + 1)

    df['Complexity * Time'] = df['Complexity Average'] * df['Play Time']
    df['Age * Complexity'] = df['Min Age'] * df['Complexity Average']
    df['Players * Complexity'] = df['Avg Players'] * df['Complexity Average']
    df['Time per Player'] = df['Play Time'] / (df['Avg Players'] + 1)

    df['Log Users'] = np.log1p(df['Users Rated'])
    df['Log Owned'] = np.log1p(df['Owned Users'])
    df['Log Time'] = np.log1p(df['Play Time'])
    df['Log BGG'] = np.log1p(df['BGG Rank'])
    df['Log Popularity'] = np.log1p(df['Popularity'])

    df['Complexity2'] = df['Complexity Average'] ** 2
    df['Time2'] = df['Play Time'] ** 2
    df['Sqrt Users'] = np.sqrt(df['Users Rated'] + 1)
    df['Sqrt Owned'] = np.sqrt(df['Owned Users'] + 1)

    df['Exp Complexity'] = np.exp(-df['Complexity Average'])
    df['Exp Years'] = np.exp(-df['Years Since'] / 50)

    df['Year Rank'] = df['Year Published'].rank(pct=True)
    df['Users Rank'] = df['Users Rated'].rank(pct=True)
    df['Complexity Rank'] = df['Complexity Average'].rank(pct=True)

    df['Age Category'] = pd.cut(df['Min Age'], bins=[0, 6, 10, 14, 100],
                                 labels=['Kids', 'Preteen', 'Teen', 'Adult'])
    df['Time Category'] = pd.cut(df['Play Time'], bins=[0, 30, 60, 120, 300, 1000],
                                  labels=['Very Short', 'Short', 'Medium', 'Long', 'Epic'])
    df['Complexity Category'] = pd.cut(df['Complexity Average'], bins=[0, 1.5, 2.5, 3.5, 5],
                                        labels=['Light', 'Medium Light', 'Medium Heavy', 'Heavy'])
    df['Decade'] = (df['Year Published'] // 10) * 10
    df['Is Modern'] = (df['Year Published'] >= 2010).astype(int)
    df['Is Classic'] = (df['Year Published'] <= 1990).astype(int)

    df = pd.get_dummies(df, columns=['Age Category', 'Time Category', 'Complexity Category'],
                         prefix=['age', 'time', 'comp'])

    return df

train_processed = create_advanced_features(train_df)
test_processed = create_advanced_features(test_df)

exclude = ['ID', 'Name', 'Mechanics', 'Domains', 'Rating Average', 'index']
features = [col for col in train_processed.columns if col not in exclude]

X = train_processed[features]
y = train_processed['Rating Average']
X_test = test_processed[features]

print(f"Features: {len(features)}")

Features: 50


In [54]:
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [58]:
models = {
    'lgbm': LGBMRegressor(n_estimators=1500, max_depth=8, learning_rate=0.015,
                          num_leaves=40, subsample=0.7, colsample_bytree=0.7,
                          reg_alpha=0.05, reg_lambda=0.05, min_child_samples=10,
                          random_state=42, verbose=-1, n_jobs=-1),
    'xgb': XGBRegressor(n_estimators=1500, max_depth=7, learning_rate=0.015,
                        subsample=0.7, colsample_bytree=0.7, reg_alpha=0.05,
                        reg_lambda=0.05, min_child_weight=3, random_state=42,
                        verbosity=0, n_jobs=-1),
    'rf': RandomForestRegressor(n_estimators=500, max_depth=12, min_samples_split=6,
                                min_samples_leaf=3, random_state=42, n_jobs=-1),
    'catboost': CatBoostRegressor(iterations=1000, depth=6, learning_rate=0.03,
                                   l2_leaf_reg=3, random_seed=42, verbose=0)
}

val_preds = np.zeros((len(X_val), len(models)))
test_preds = np.zeros((len(X_test_scaled), len(models)))

for i, (name, model) in enumerate(models.items()):
    print(f"train {name}")
    model.fit(X_train, y_train)
    val_preds[:, i] = model.predict(X_val)
    test_preds[:, i] = model.predict(X_test_scaled)
    mse = mean_squared_error(y_val, val_preds[:, i])
    print(f"{name} MSE: {mse:.6f}")

train lgbm
lgbm MSE: 0.043115
train xgb
xgb MSE: 0.042337
train rf
rf MSE: 0.065555
train catboost
catboost MSE: 0.046573


In [59]:
mses = [mean_squared_error(y_val, val_preds[:, i]) for i in range(4)]
weights_by_mse = (1 / np.array(mses)) / sum(1 / np.array(mses))

blended_val = np.dot(val_preds, weights_by_mse)
blended_mse = mean_squared_error(y_val, blended_val)

meta_model = LGBMRegressor(n_estimators=300, max_depth=4, learning_rate=0.03,
                           random_state=42, verbose=-1)
meta_model.fit(val_preds, y_val)
meta_val_pred = meta_model.predict(val_preds)
meta_mse = mean_squared_error(y_val, meta_val_pred)

if meta_mse < blended_mse:
    final_predictions = meta_model.predict(test_preds)
    print(f"Using meta-model (MSE: {meta_mse:.6f})")
else:
    final_predictions = np.dot(test_preds, weights_by_mse)
    print(f"Using blending (MSE: {blended_mse:.6f})")

Using meta-model (MSE: 0.034682)


In [61]:
real_mean = y.mean()
real_std = y.std()
pred_mean = final_predictions.mean()
pred_std = final_predictions.std()

final_predictions = (final_predictions - pred_mean) / pred_std
final_predictions = final_predictions * real_std + real_mean
final_predictions = np.clip(final_predictions, 1, 10)
final_predictions = np.round(final_predictions, 3)

submission = pd.DataFrame({
    'index': sample_sub['index'],
    'Rating Average': final_predictions
})

submission.to_csv('submission.csv', index=False)

In [62]:
!kaggle competitions submit -c games-rating -f submission.csv  -m "fifth"

100% 53.0k/53.0k [00:00<00:00, 197kB/s]
Successfully submitted to Games Rating